In [1]:
# 파이썬에게 "이 도구들을 꺼내와!" 라고 명령하는 부분입니다. (라이브러리 불러오기)
import pandas as pd
import numpy as np
import importlib
import glob
import os

print(f"Pandas 버전 확인: {pd.__version__}")
print(f"Scikit-Learn 버전 확인: {importlib.import_module('sklearn').__version__}")

# [업데이트] 노트북 파일이 Python_Analytics 폴더 안으로 이동했으므로,
# 상위 폴더("..")로 한 칸 나간 뒤 "SimulationLogs/MLCluster" 폴더 안에 있는 모든 .csv 파일을 찾아오라고 명령합니다.
file_pattern = os.path.join("..", "SimulationLogs", "MLCluster", "*", "*.csv")
csv_files = glob.glob(file_pattern)

print(f"발견된 ML 데이터 파일 수: {len(csv_files)}개")

Pandas 버전 확인: 2.3.3
Scikit-Learn 버전 확인: 1.7.2
발견된 ML 데이터 파일 수: 5개


In [10]:
# ==========================================
# [실행 메모장] 데이터 불러오기 및 3단계 교차 검증
# ==========================================

# 1. 방금 우리가 만든 DataLoader.py 파일에서 DataLoader 설계도를 가져옵니다!
from DataLoader import DataLoader
import pandas as pd

# 2. 객체 생성 (기계를 조립합니다)
loader = DataLoader()

# 3. 데이터 병합 (기계를 작동시킵니다)
raw_df = loader.load_and_merge_csvs()

# 4. 필터링 (2라운드 이하는 노이즈이므로, 3라운드 이상만 남깁니다)
filtered_df = loader.filter_noise(min_rounds=3)

# 5. 스케일링 (학습에 들어갈 X_scaled 정답지와, 유저 번호 sim_ids를 분리해서 받습니다)
X_scaled, sim_ids = loader.get_scaled_features()

# ==========================================
# [기획자 전용 시각화] 원본 -> 평균 -> 스케일 결과 순차 출력
# ==========================================

print("=======================================================")
print("🚨 0. 노이즈 포함 원본 데이터 (Raw Data) - 필터링 전 모든 기록과 칼럼")
print("=======================================================")
# [추가됨] 필터링 되기 전, 1~2라운드 종료 데이터와 Sim_ID까지 모두 포함된 진짜 날것입니다.
display(raw_df.head())

print("=======================================================")
print("📊 1. 원본 데이터 (Raw Data) - 스케일링 되기 전 순수 수치")
print("=======================================================")
# 학습에 사용된 10개의 피처만 뽑아서 상위 5개를 예쁘게 출력합니다.
display(filtered_df[loader.feature_columns].head())

print("\n=======================================================")
print("🎯 2. 각 필드의 평균치 (Mean) - 0.0의 기준점이 되는 수치")
print("=======================================================")
# pandas의 .mean() 함수를 사용해 각 칼럼의 평균을 구합니다.
mean_values = filtered_df[loader.feature_columns].mean()
# 보기 좋게 데이터프레임(표)으로 감싸서 출력합니다.
display(pd.DataFrame(mean_values, columns=['전체 평균값']))

print("\n=======================================================")
print("🧬 3. 정규화된 데이터 (Scaled Data) - 평균을 0.0으로 찌그러뜨린 결과")
print("=======================================================")
display(X_scaled.head())

[1/3] 파일 로드 완료: 총 5개의 파일을 병합했습니다. (총 전투 기록: 500개)
[2/3] 필터링 완료: 428개의 유효하지 않은 짧은 전투(노이즈)가 제거되었습니다. (남은 데이터: 72개)
[3/3] 정규화 완료: 데이터 스케일링 성공. 머신러닝 학습 준비 완료.
🚨 0. 노이즈 포함 원본 데이터 (Raw Data) - 필터링 전 모든 기록과 칼럼


,Sim_ID,Total_Rounds,Alive_Ratio,Remaining_HP_Ratio,YinYang_Deviation,Skill_Aggressive,Skill_Heal,Skill_Utility,Skill_Defensive,Turn_Skip_Ratio,Corrosion_Revert_Ratio
0,1,2,1.0,0.991,0.0,0.8,0.0,0.4,0.0,0.0,0.0
1,2,2,1.0,1.000,0.0,0.6,0.0,0.6,0.0,0.0,0.0
2,3,1,1.0,0.989,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,4,2,1.0,0.991,0.0,0.8,0.0,0.6,0.0,0.0,0.0
4,5,1,1.0,0.986,0.0,1.0,0.0,0.0,0.0,0.0,0.0


📊 1. 원본 데이터 (Raw Data) - 스케일링 되기 전 순수 수치


,Total_Rounds,Alive_Ratio,Remaining_HP_Ratio,YinYang_Deviation,Skill_Aggressive,Skill_Heal,Skill_Utility,Skill_Defensive,Turn_Skip_Ratio,Corrosion_Revert_Ratio
27,4,1.0,0.959,11.3,0.385,0.077,0.615,0.0,0.0,0.0
101,3,1.0,0.908,10.0,0.556,0.000,0.556,0.0,0.0,0.0
103,3,1.0,0.892,0.0,0.556,0.000,0.667,0.0,0.0,0.0
120,3,1.0,0.975,0.0,0.556,0.000,0.667,0.0,0.0,0.0
128,3,1.0,0.975,0.0,0.778,0.000,0.556,0.0,0.0,0.0



🎯 2. 각 필드의 평균치 (Mean) - 0.0의 기준점이 되는 수치


,전체 평균값
Total_Rounds,3.027778
Alive_Ratio,1.000000
Remaining_HP_Ratio,0.941597
YinYang_Deviation,0.712500
Skill_Aggressive,0.591764
Skill_Heal,0.091431
Skill_Utility,0.398944
Skill_Defensive,0.000000
Turn_Skip_Ratio,0.000000
Corrosion_Revert_Ratio,0.000000



🧬 3. 정규화된 데이터 (Scaled Data) - 평균을 0.0으로 찌그러뜨린 결과


,Total_Rounds,Alive_Ratio,Remaining_HP_Ratio,YinYang_Deviation,Skill_Aggressive,Skill_Heal,Skill_Utility,Skill_Defensive,Turn_Skip_Ratio,Corrosion_Revert_Ratio
27,5.916080,0.0,0.374770,4.053756,-3.420565,-0.135421,1.800286,0.0,0.0,0.0
101,-0.169031,0.0,-0.723518,3.556010,-0.591654,-0.858014,1.308668,0.0,0.0,0.0
103,-0.169031,0.0,-1.068078,-0.272803,-0.591654,-0.858014,2.233577,0.0,0.0,0.0
120,-0.169031,0.0,0.719330,-0.272803,-0.591654,-0.858014,2.233577,0.0,0.0,0.0
128,-0.169031,0.0,0.719330,-0.272803,3.080967,-0.858014,1.308668,0.0,0.0,0.0
